# Centralize Baseline
### ResNet18 on CIFAR-100

In [1]:
# =============================================================================
# Centralize Baseline: ResNet18 on CIFAR-10
# - Single-model, end-to-end training on union of all data
# - Logs acc_train, acc_test, round_time_sec, mean_FgT_loss (per epoch)
# - Saves CSV + for comparison plots
# ============================================================================

import copy
import math  # Cosine learning-rate schedule to stabilize training
import os
import random
import time  # Measuring wall-clock time per epoch to compare efficiency

# ============================================================================
#                               Matplotlib
# ============================================================================
import matplotlib

# ============================================================================
#                                Third-party
# ============================================================================
import numpy as np
import pandas as pd
import torch  # Core deep learning tensor library (GPU/CPU compute)
import torch.nn.functional as F  # Functional ops like softmax
from torch import nn  # Neural network layers/losses
from torch.utils.data import DataLoader, Dataset  # Batching datasets
from torchvision import (  # CIFAR datasets + transforms + ResNet
    datasets,
    models,
    transforms,
)

matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ============================================================================
#                         Colored terminal output
# ============================================================================
# To print in color -------test/train of the client side
def prRed(skk):
    print("\033[91m {}\033[00m".format(skk))


def prGreen(skk):
    print("\033[92m {}\033[00m".format(skk))


def _sync_cuda():
    """
    Synchronize CUDA to get accurate timing.
    Compare wall time across methods.
    """
    if torch.cuda.is_available():
        torch.cuda.synchronize()


# ============================================================================
#                         Configuration
# ============================================================================
PROGRAM_NAME = "Centralize Baseline - CIFAR-100"
DATASET_NAME = "CIFAR100"  # Set toggle between CIFAR-10 and CIFAR-100
DATA_ROOT = "data"  # Set where torchvision stores CIFAR data
EPOCHS = 200
BATCH_SIZE = 256
LR = 0.0001
MOMENTUM = 0.9  # Smoothing/Accelerating SGD undates
WEIGHT_DECAY = 5e-4  # L2 Regularization
LABEL_SMOOTH = 0.0
NUM_WORKERS = 4
SEED = 1234
WARMUP_EPOCHS = 0


# ============================================================================
#                             Toggle Cosine
# ============================================================================
cosine_schedule = False       # False = use a constant LR
if cosine_schedule:
    SCHEDULER = "cosine"


# ============================================================================
#                             Reproduce
# ============================================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)  # Seed CPU RNG for Pytorch ops
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)  # Seed all GPUs for DataParallel)
    torch.backends.cudnn.deterministic = True  # Pick deterministic algorithms
    torch.backends.cudnn.benchmark = False  # Turn off autotuner for derterminism


# ============================================================================
#                           Device Selection
# ============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

print(f"---------{PROGRAM_NAME}----------")


# ============================================================================
#                           Data & Transforms
#               Setting mean/std between CIFAR-10 & CIFAR-100
#    Normallizing inpits accelerates convergence and stabilizes training
# ============================================================================
CIFAR_MEAN = {
    "CIFAR10": [0.4914, 0.4822, 0.4465],
    "CIFAR100": [0.5071, 0.4867, 0.4408],
}[DATASET_NAME]
CIFAR_STD = {
    "CIFAR10": [0.2470, 0.2435, 0.2616],
    "CIFAR100": [0.2675, 0.2565, 0.2761],
}[DATASET_NAME]


# Training time
train_tfms = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),  # Convert PIL Image -> torch.FloatTensor in [0,1]
        transforms.Normalize(
            CIFAR_MEAN, CIFAR_STD
        ),  # Normalize to zero-mean/unit-var per channel
    ]
)

# Test-time we only normalize, no random, to get stable evaluation.
test_tfms = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ]
)


# ============================================================================
#                               DataLoders
# ============================================================================
# Download CIFAR datasets
if DATASET_NAME == "CIFAR10":
    train_set = datasets.CIFAR10(
        root=DATA_ROOT, train=True, download=True, transform=train_tfms
    )
    test_set = datasets.CIFAR10(
        root=DATA_ROOT, train=False, download=True, transform=test_tfms
    )
    NUM_CLASSES = 10
else:
    train_set = datasets.CIFAR100(
        root=DATA_ROOT, train=True, download=True, transform=train_tfms
    )
    test_set = datasets.CIFAR100(
        root=DATA_ROOT, train=False, download=True, transform=test_tfms
    )
    NUM_CLASSES = 100


# Shuffling, batching, perfetching
train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_set,
    batch_size=BATCH_SIZE,
    shuffle=False,  # No shuffle at test time
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


# ============================================================================
#                      Model (CIFAR ResNet-18 - 32 x 32)
#                    3 x 3, stride-1 conv, removing maxpool
# ============================================================================
def build_cifar_resnet18(num_classes: int) -> nn.Module:
    m = models.resnet18(weights=None)
    m.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()  # Remove large downsampling for 32x32 inputs
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m


model = build_cifar_resnet18(NUM_CLASSES).to(device)  # Move model to GPU/CPU


# If multiple GPUs are visible, DataParallel distributes batches across them.
if torch.cuda.device_count() > 1:
    print(f"We use {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

# Cross-entropy - LABEL_SMOOTHING is now set to 0.0 to match SL/SFLV2
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH).to(device)

# SGD with momentum + weight decay
optimizer = torch.optim.SGD(
    model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY
)


# =============================================================================
#                 Schedule: warmup (epoches) + cosine
#             if cosine_schedule = True will trigger below
#                   False = use the constant LR
# =============================================================================
if cosine_schedule:
    if WARMUP_EPOCHS >= EPOCHS:
        raise ValueError("WARMUP_EPOCHS must be less than (<) EPOCHS")
    
    # Linear warmup for first WARMUP_EPOCHS epochs
    warmup = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=1e-8,  # (LinearLR doesn't allow start_factor=0.0 change it to 1e-8 (0.000000001))
        end_factor=1.0,
        total_iters=WARMUP_EPOCHS,
    )
    
    # Cosine for the remaining epochs
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS - WARMUP_EPOCHS,  # number of cosine steps after warmup
        eta_min=0.0,
    )
    
    # Chain: warmup -> cosine
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS]
    )


# =============================================================================
#                 Matrix for train/test accuracy logging
# =============================================================================
def top1_accuracy(logits: torch.Tensor, y: torch.Tensor) -> float:
    pred = logits.argmax(dim=1)
    return (pred == y).float().mean().item() * 100.0


# Evaluation loop
# Disable grad for speed
# Aggregate loss and accuracy
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y).item()
        loss_sum += loss * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return (loss_sum / max(1, total), 100.0 * correct / max(1, total))


# Training epoch over the training set.
def train_one_epoch(
    model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer
):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)  # set_to_none; avoids writing zeros
        logits = model(x)  # Forward pass
        loss = criterion(logits, y)  # Compute supervised loss
        loss.backward()  # Backprop to compute gradients
        optimizer.step()  # Update parameters
        loss_sum += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return (loss_sum / max(1, total), 100.0 * correct / max(1, total))


# =============================================================================
#                         Windowed forgetting metric
# Splitting the test set into EPOCHS disjoint windows.
# (1) - After each epoch e, update the best-so-far loss on window e.
# (2) - Measuring how much the loss on all past windows has risen vs there historical minimum.
# =============================================================================
class DatasetSplit(Dataset):
    def __init__(self, base: Dataset, idxs):
        self.base, self.idxs = base, list(idxs)

    def __len__(self):
        return len(self.idxs)

    def __getitem__(self, i: int):
        return self.base[self.idxs[i]]


# Build disjoint validation loaders per epoch index (round-robin partitioning).
val_loaders = {}
idx_all = np.arange(len(test_set))
for t in range(EPOCHS):
    idxs = idx_all[t::EPOCHS]  # window t gets items t, t+EPOCHS, t+2*EPOCHS, ...
    val = DatasetSplit(test_set, idxs)
    val_loaders[t] = DataLoader(
        val,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )


best_loss_per_window = [
    float("inf")
] * EPOCHS  # Track each window's best (min) loss so far
forgetting_log = []  # Store per-epoch mean_FgT_loss values for saving/plotting


assert all(len(ld.dataset) > 0 for ld in val_loaders.values())


# Evaluate average loss on a particular window.
@torch.no_grad()
def eval_loss_on_window(model: nn.Module, t: int) -> float:
    loader = val_loaders[t]
    if len(loader.dataset) == 0:
        return 0.0

    model.eval()

    total = 0
    loss_sum = 0.0

    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y).item()  # <--- use global criterion

        # Skip non-finite losses to avoid poisoning FgT
        if not math.isfinite(loss):
            continue

        loss_sum += loss * y.size(0)
        total += y.size(0)

    if total == 0:
        # No valid batches for this window; treat as "no signal"
        return float("nan")

    return loss_sum / total


# =============================================================================
#                         Training loop
# =============================================================================
# Plotting accuracy vs epoch
acc_train_hist = []
acc_test_hist = []

round_time_hist = []  # Efficiency
mean_fgt_hist = []  # Forgetting


for epoch in range(EPOCHS):
    _sync_cuda()
    t0 = time.perf_counter()  # Start accurate timer (after syncing GPU)


    # ================= Cosine Toggle  =====================================
    if cosine_schedule:
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)  # Train one epoch
        test_loss, test_acc = evaluate(model, test_loader)  # Evaluate on test

        scheduler.step()  # epoch-level step
    else:                 # Use constant LR
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
        test_loss, test_acc = evaluate(model, test_loader) # Evaluate on test
    

    _sync_cuda()
    dt = time.perf_counter() - t0  # Stop timer

    # Pretty console logs help when scanning SLURM outputs
    cur_lr = optimizer.param_groups[0]["lr"]

    if cosine_schedule:
        prRed(
            f"Train => Epoch {epoch:03d}  Acc: {train_acc:6.3f}  Loss: {train_loss:7.4f}  LR: {scheduler.get_last_lr()[0]:.5f}"
        )
        prGreen(
            f"Test  => Epoch {epoch:03d}  Acc: {test_acc:6.3f}  Loss: {test_loss:7.4f}  Time(s): {dt:7.2f}"
        )
    else:
        prRed(
            f"Train => Epoch {epoch:03d}  Acc: {train_acc:6.3f}  Loss: {train_loss:7.4f}  LR: {cur_lr:.5f}"
        )
        prGreen(
            f"Test  => Epoch {epoch:03d}  Acc: {test_acc:6.3f}  Loss: {test_loss:7.4f}  Time(s): {dt:7.2f}"
        )

    
    # --- forgetting statistic (No NaN/Inf) ---
    t = epoch  # current window id (0-based)

    # 1) update historical minimum for the *current* window,
    #    but only if the loss is finite.
    L_cur_t = eval_loss_on_window(model, t)
    if math.isfinite(L_cur_t):
        if L_cur_t < best_loss_per_window[t]:
            best_loss_per_window[t] = L_cur_t
    # else: skip update; this window still has no valid baseline

    # 2) forgetting on all PAST windows vs their minima
    per_win_fgt = []
    for t0 in range(t):
        L_now = eval_loss_on_window(model, t0)
        best = best_loss_per_window[t0]

        # only use windows where both current and best are finite
        if not (math.isfinite(L_now) and math.isfinite(best)):
            continue

        Fgt = L_now - best  # allow negatives
        per_win_fgt.append(Fgt)

    mean_fgt = float(np.mean(per_win_fgt)) if per_win_fgt else 0.0

    # --- Log history vectors ---
    acc_train_hist.append(train_acc)
    acc_test_hist.append(test_acc)
    round_time_hist.append(dt)
    mean_fgt_hist.append(mean_fgt)

    print(f"Forgetting : Epoch {epoch}, mean_FgT_loss = {mean_fgt:.6f}")
    print("==========================================================")

print("Training and Evaluation completed!")


# =============================================================================
#                                  Save logs
# =============================================================================
# Keep lengths aligned (paranoia)
n = min(
    len(acc_train_hist), len(acc_test_hist), len(round_time_hist), len(mean_fgt_hist)
)
round_idx = list(range(1, n + 1))

df = pd.DataFrame(
    {
        "round": round_idx,
        "acc_train": acc_train_hist[:n],
        "acc_test": acc_test_hist[:n],
        "round_time_sec": round_time_hist[:n],
        "mean_FgT_loss": mean_fgt_hist[:n],
    }
)

csv_name = f"{PROGRAM_NAME}.csv"
df.to_csv(csv_name, index=False)
print("Saved:", csv_name)

# =============================================================================
#                         Program Completed
# =============================================================================

NVIDIA H200
---------Centralize Baseline - CIFAR-100----------


100%|██████████| 169M/169M [00:07<00:00, 21.3MB/s] 


 Train => Epoch 000  Acc:  8.462  Loss:  4.0378  LR: 0.01000
 Test  => Epoch 000  Acc: 13.450  Loss:  3.6269  Time(s):    7.77
Forgetting : Epoch 0, mean_FgT_loss = 0.000000
 Train => Epoch 001  Acc: 17.746  Loss:  3.4121  LR: 0.01000
 Test  => Epoch 001  Acc: 19.840  Loss:  3.2866  Time(s):    5.14
Forgetting : Epoch 1, mean_FgT_loss = -0.342540
 Train => Epoch 002  Acc: 24.826  Loss:  3.0055  LR: 0.01000
 Test  => Epoch 002  Acc: 26.260  Loss:  2.9454  Time(s):    7.13
Forgetting : Epoch 2, mean_FgT_loss = -0.450694
 Train => Epoch 003  Acc: 30.890  Loss:  2.7023  LR: 0.01000
 Test  => Epoch 003  Acc: 31.780  Loss:  2.6549  Time(s):    7.15
Forgetting : Epoch 3, mean_FgT_loss = -0.541102
 Train => Epoch 004  Acc: 35.490  Loss:  2.4661  LR: 0.01000
 Test  => Epoch 004  Acc: 35.470  Loss:  2.5042  Time(s):    7.15
Forgetting : Epoch 4, mean_FgT_loss = -0.495273
 Train => Epoch 005  Acc: 39.806  Loss:  2.2717  LR: 0.01000
 Test  => Epoch 005  Acc: 39.780  Loss:  2.3102  Time(s):    7.14